In [1]:
import sys
sys.path.append("..")            # 저장소 루트 (project 패키지)
sys.path.append("../scripts")    # eval_silver 변환 함수 재사용

In [2]:
from pathlib import Path

BENCH_ID = "OpenSLR_SLR40_Zeroth_ko_general_clean"
SILVER   = "/data/ASR/BENCHMARK/SILVER/OpenSLR_SLR40_Zeroth_ko/transcript.jsonl"
MODEL    = "openai/whisper-small"
DEVICE   = "cuda:2"              # 실행 직전 nvidia-smi 로 빈 GPU 확인 후 지정

OUT_DIR = Path(f"../BENCHMARK/results/whisper_small__{BENCH_ID}")

In [3]:
from eval_silver import convert_silver

conv = OUT_DIR / "_silver_converted" / f"{BENCH_ID}.jsonl"
n = convert_silver(Path(SILVER), conv, corpus_id=BENCH_ID)
print(f"{n} samples → {conv}")

457 samples → ../BENCHMARK/results/whisper_small__OpenSLR_SLR40_Zeroth_ko_general_clean/_silver_converted/OpenSLR_SLR40_Zeroth_ko_general_clean.jsonl


In [4]:
from project.data.adapters.whisper import build_predict_fn

predict_fn = build_predict_fn(
    MODEL, backbone=MODEL,
    language="ko", task="transcribe",
    beam_size=5, batch_size=16, device=DEVICE,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [5]:
from project.evaluation import evaluate_on_benchmark_suite

results = evaluate_on_benchmark_suite(
    model_name=f"whisper_small__{BENCH_ID}",
    predict_fn=predict_fn,
    benchmark_paths={BENCH_ID: conv},
    out_dir=OUT_DIR,
    batch_size=16,
)
results[BENCH_ID]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take p

CerResult(cer=10.576319095477386, scer=9.48257984238905, wer=32.04336696280681, samples=457, per_sample_cer=[5.88235294117647, 4.25531914893617, 16.176470588235293, 2.5974025974025974, 7.317073170731707, 2.083333333333333, 2.666666666666667, 2.127659574468085, 0.0, 10.16949152542373, 1.3888888888888888, 5.154639175257731, 21.951219512195124, 2.8169014084507045, 30.681818181818183, 6.756756756756757, 34.146341463414636, 0.0, 8.0, 6.25, 6.8181818181818175, 17.391304347826086, 6.25, 7.2727272727272725, 10.16949152542373, 3.8461538461538463, 8.695652173913043, 12.244897959183673, 5.263157894736842, 3.076923076923077, 1.4084507042253522, 8.235294117647058, 26.31578947368421, 10.344827586206897, 12.121212121212121, 2.3255813953488373, 15.09433962264151, 4.477611940298507, 6.756756756756757, 2.631578947368421, 4.761904761904762, 3.7735849056603774, 2.3255813953488373, 4.878048780487805, 0.0, 7.4074074074074066, 1.6129032258064515, 11.11111111111111, 15.625, 8.0, 20.33898305084746, 7.042253521

In [6]:
print((OUT_DIR / "evaluation_report.txt").read_text(encoding="utf-8"))

📊 ASR Evaluation Report — whisper_small__OpenSLR_SLR40_Zeroth_ko_general_clean
   Date: 2026-06-16T10:29:25

## 1. Benchmark Set Results (한국어 CER 표준)
--------------------------------------------------------------------------------
Benchmark                                                  CER (%)   sCER (%)    Samples
--------------------------------------------------------------------------------
OpenSLR_SLR40_Zeroth_ko_general_clean                        10.58       9.48        457
--------------------------------------------------------------------------------
Weighted Average                                             10.58                   457

## 2. Slice Analysis (메타 필드별)
--------------------------------------------------------------------------------

### OpenSLR_SLR40_Zeroth_ko_general_clean
  [by age_group]
  value                   CER (%)    samples
  unknown                   10.58        457
  [by gender]
  value                   CER (%)    samples
  female           

In [7]:
import json
import pandas as pd
import jiwer

lines = (OUT_DIR / BENCH_ID / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
df = pd.DataFrame(json.loads(l) for l in lines if l)

df["cer"] = [
    jiwer.cer(r, h) * 100 if r else float("nan")
    for r, h in zip(df["text_normalized"], df["prediction_normalized"])
]

worst = df.sort_values("cer", ascending=False).head(20)
for _, row in worst.iterrows():
    print(f"[CER {row.cer:5.1f}] 정답: {row.text_normalized}")
    print(f"             예측: {row.prediction_normalized}\n")

[CER  88.2] 정답: 차 문이 종잇장처럼 얇지 않으니 문 두께 스물 센티미터를 빼면 실제 승 하차 여유 공간은 스물 센티미터라는 계산이 나옵니다
             예측: 계속 찍는 게 녹음되고 있는 거야 조용히 하면

[CER  71.2] 정답: 반면 하종강 성공회대 노동아카데미 주임교수는 인간에 대한 이해 부족이 낳은 현상이라며 실직하는 노동자의 비인간적 고통을 생각해야 한다고 강조했다
             예측: 반면 하중간 성공에 대해 노동 아카데미 주인 교수는 인간에 대해

[CER  56.5] 정답: 삼 분기에만 매출 일 조 이천 구백 십 구 억원 영업이익 육백 십 팔 억원을 올렸다
             예측: 3분기 매출 1초 2919원 영업이 618억원을 올렸다

[CER  48.0] 정답: 경남에서도 열 여덟 개 시 군 가운데 열 한 개 시 군만 월 삼 만 십 만원의 수당을 준다
             예측: 경남에서도 18개씩은 가운데 11개씩은 만 5월 30만원에 수당을 준다

[CER  46.7] 정답: 국립 박물관의 연구원 테레자 본토르치히는 자신의 저서 믿음 때문에 투옥되다 아우슈비츠 강제 수용소의 여호와의 증인 에서 이렇게 썼습니다
             예측: 국립박물관이 연구한 테레자 본토르 지인은 자신에 져서 믿음 때문에 투옥되다 아우스비츠 강재수요

[CER  46.2] 정답: 회수대상은 이천 십 육 년 시 월 일 일 에서 이천 십 육 년 십 일 월 십 구 일 내로 유통기한이 표시된 제품들이다
             예측: 해수 대상은 2016년 10월 1일인에서 2016년 11월 19일 내로 유통기관이 표시된 제품들이다

[CER  37.9] 정답: 노르웨이는 육아휴직 마흔 일곱 주를 사용하면 임금의 백 퍼센트를 쉰 일곱 주를 사용하면 임금의 팔십 퍼센트를 지급합니다
             예측: 노르웨이는 유가 휴직 47주를 사용하면 임금의 100 를 57주를 사용하면 임금의 80 를 지극합니다

[CER  

In [8]:
import re, jiwer

def strip_spaces_in_numbers(t):
    # "일 조 이천 구백" 처럼 띄어쓴 한글 숫자를 비교에서 덜 불리하게:
    # 간단버전 — 숫자 인접 공백 제거로 표기차 일부 흡수
    return re.sub(r"(?<=\d)\s+(?=\d)", "", t)

# 더 정확히는 한글 수사 ↔ 아라비아 변환이 필요하지만,
# 우선 숫자가 포함된 발화와 아닌 발화의 CER을 분리해서 영향도부터 확인:
has_num = df["text_normalized"].str.contains(r"[0-9]|영|일|이|삼|사|오|육|칠|팔|구|십|백|천|만|조")
print(f"숫자 포함 발화: {has_num.sum()}개, CER {df.loc[has_num,'cer'].mean():.1f}%")
print(f"숫자 없는 발화: {(~has_num).sum()}개, CER {df.loc[~has_num,'cer'].mean():.1f}%")

숫자 포함 발화: 433개, CER 10.4%
숫자 없는 발화: 24개, CER 6.1%
